# Remaining Hindi Manual Collection & Re-Split (v3)

**Purpose**: Collect ~20M additional manual Hindi tokens to push:
- Manual share from **16.9% → ≥ 20%**
- Total tokens from **460.7M → ≥ 500M**

**What's new in v3**:
- Resilient Drive remount before heavy I/O
- Corrupt JSONL line handling (skip bad lines)
- Hindi Wiktionary + Wikivoyage dumps as extra manual sources
- Much deeper archive pagination (500+ pages per site)
- More news portals

---
## Cell 1: Setup & Google Drive Mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q trafilatura sentencepiece requests beautifulsoup4 tabulate lxml

import json, hashlib, re, random, bz2, os, gc, sys, shutil, time, math, collections, requests
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import urljoin, urlparse
from bs4 import BeautifulSoup
import trafilatura
import sentencepiece as spm

LANGUAGE = 'hindi'
DRIVE_BASE = '/content/drive/MyDrive/24KaratAu/hindi'
MANUAL_DIR = f'{DRIVE_BASE}/cleaned/manual'
NONWEB_DIR = f'{DRIVE_BASE}/cleaned/automated_nonweb'
WEB_DIR    = f'{DRIVE_BASE}/cleaned/automated_web'
SPLITS_DIR = f'{DRIVE_BASE}/splits'
TOKENIZER_DIR = f'{DRIVE_BASE}/tokenizer'

for d in [MANUAL_DIR, NONWEB_DIR, WEB_DIR, SPLITS_DIR, TOKENIZER_DIR]:
    os.makedirs(d, exist_ok=True)

CHARS_PER_TOKEN = 3.48
MIN_CHARS = 80
MIN_DEV_RATIO = 0.50
MAX_LATIN_RATIO = 0.10
SHARD_SIZE = 5000

def devanagari_ratio(text):
    if not text: return 0.0
    return sum(1 for c in text if '\u0900' <= c <= '\u097F') / len(text)

def latin_ratio(text):
    if not text: return 0.0
    return sum(1 for c in text if 'a' <= c.lower() <= 'z') / len(text)

def clean_web_text(text):
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'\S+@\S+\.\S+', '', text)
    text = re.sub(r'&[a-zA-Z]+;', ' ', text)
    text = re.sub(r'&#\d+;', ' ', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def text_hash_int(text):
    h = hashlib.sha256(text.encode('utf-8')).hexdigest()
    return int(h[:16], 16)

WIKI_MARKUP = [
    (re.compile(r'\{\{[^}]*\}\}'), ''),
    (re.compile(r'\[\[Category:[^\]]*\]\]'), ''),
    (re.compile(r'\[\[श्रेणी:[^\]]*\]\]'), ''),
    (re.compile(r'\[\[वर्ग:[^\]]*\]\]'), ''),
    (re.compile(r'\[\[[^|\]]*\|([^\]]*)\]\]'), r'\1'),
    (re.compile(r'\[\[([^\]]*)\]\]'), r'\1'),
    (re.compile(r'\[https?://[^\s\]]*\s*([^\]]*)\]'), r'\1'),
    (re.compile(r'\[https?://[^\]]*\]'), ''),
    (re.compile(r"'{2,5}"), ''),
    (re.compile(r'<ref[^>]*>.*?</ref>', re.DOTALL), ''),
    (re.compile(r'<ref[^/>]*/>'), ''),
    (re.compile(r'<[^>]+>'), ''),
    (re.compile(r'&[a-zA-Z]+;'), ' '),
    (re.compile(r'\[\d+\]'), ''),
]
TABLE_PAT = [
    (re.compile(r'^\s*\{\|.*$', re.MULTILINE), ''),
    (re.compile(r'^\s*\|\}.*$', re.MULTILINE), ''),
    (re.compile(r'^\s*\|-.*$', re.MULTILINE), ''),
    (re.compile(r'^\s*!.*$', re.MULTILINE), ''),
    (re.compile(r'^\|.*$', re.MULTILINE), ''),
]
SKIP_SEC = {'सन्दर्भ', 'संदर्भ', 'बाहरी कड़ियाँ', 'बाहरी कड़ी', 'श्रेणी',
            'इन्हें भी देखें', 'यह भी देखें', 'स्रोत', 'टिप्पणी', 'नोट',
            'references', 'external links', 'see also', 'notes'}

def clean_wiki(text):
    for p, r in TABLE_PAT: text = p.sub(r, text)
    for p, r in WIKI_MARKUP: text = p.sub(r, text)
    lines, result, skip = text.split('\n'), [], False
    for line in lines:
        s = line.strip().lower()
        if s in SKIP_SEC: skip = True; continue
        if re.match(r'^={2,}.*={2,}$', s): skip = False
        if not skip: result.append(line)
    text = '\n'.join(result)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def safe_json_read(filepath):
    """Read a JSONL file, skipping corrupt/truncated lines."""
    docs = []
    bad = 0
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            for line in f:
                if not line.strip(): continue
                try:
                    docs.append(json.loads(line))
                except (json.JSONDecodeError, UnicodeDecodeError):
                    bad += 1
    except Exception as e:
        print(f'    ⚠️  Could not read {filepath}: {e}')
    return docs, bad

# ── Load existing state ──
man_shards = sorted(Path(MANUAL_DIR).glob('*.jsonl'))
auto_shards = sorted(Path(NONWEB_DIR).glob('*.jsonl'))

print(f'Loading dedup hashes from {len(man_shards)} existing manual shards...')
existing_hashes = set()
existing_manual_chars = 0
total_bad_lines = 0
for sf in man_shards:
    docs, bad = safe_json_read(sf)
    total_bad_lines += bad
    for doc in docs:
        h = doc.get('text_sha256', '')
        if h:
            try: existing_hashes.add(h if isinstance(h, int) else int(h[:16], 16) if len(h) > 16 else int(h, 16))
            except: pass
        existing_manual_chars += doc.get('clean_text_characters', len(doc.get('text', '')))

print(f'\n📊 EXISTING STATE:')
print(f'  Manual shards:    {len(man_shards)}')
print(f'  Automated shards: {len(auto_shards)}')
print(f'  Manual chars:     {existing_manual_chars:,} (~{int(existing_manual_chars / CHARS_PER_TOKEN):,} tokens)')
print(f'  Dedup hashes:     {len(existing_hashes):,}')
if total_bad_lines: print(f'  ⚠️  Skipped {total_bad_lines} corrupt lines')

MANUAL_TOKEN_TARGET = 105_000_000
current_manual_tokens = int(existing_manual_chars / CHARS_PER_TOKEN)
tokens_needed = max(0, MANUAL_TOKEN_TARGET - current_manual_tokens)
chars_needed = int(tokens_needed * CHARS_PER_TOKEN)
print(f'\n🎯 TARGET: {MANUAL_TOKEN_TARGET:,} manual tokens')
print(f'   Current: {current_manual_tokens:,} | Need: ~{tokens_needed:,} more (~{chars_needed:,} chars)')

---
## Cell 2: Extra Wikimedia Dumps (Wiktionary + Wikivoyage + Wikibooks + Wikiquote)

In [ ]:
shard_idx = len(man_shards)
new_manual_chars = 0
new_manual_docs = 0

# All Hindi Wikimedia auxiliary dumps
AUX_DUMPS = {
    'wiktionary': 'https://dumps.wikimedia.org/hiwiktionary/latest/hiwiktionary-latest-pages-articles.xml.bz2',
    'wikivoyage': 'https://dumps.wikimedia.org/hiwikivoyage/latest/hiwikivoyage-latest-pages-articles.xml.bz2',
    'wikibooks':  'https://dumps.wikimedia.org/hiwikibooks/latest/hiwikibooks-latest-pages-articles.xml.bz2',
    'wikiquote':  'https://dumps.wikimedia.org/hiwikiquote/latest/hiwikiquote-latest-pages-articles.xml.bz2',
    'wikinews':   'https://dumps.wikimedia.org/hiwikinews/latest/hiwikinews-latest-pages-articles.xml.bz2',
    'wikiversity':'https://dumps.wikimedia.org/hiwikiversity/latest/hiwikiversity-latest-pages-articles.xml.bz2',
}

aux_docs = []
for name, url in AUX_DUMPS.items():
    # Check if shards for this source already exist
    existing_for_source = list(Path(MANUAL_DIR).glob(f'manual_{name}_shard_*.jsonl'))
    if existing_for_source:
        print(f'⏭️  {name}: {len(existing_for_source)} shard(s) already exist. Skipping.')
        continue
    
    local_bz2 = f'/content/{name}.xml.bz2'
    if not os.path.exists(local_bz2):
        print(f'📥 Downloading Hindi {name} dump...')
        ret = os.system(f'wget -q --show-progress -O {local_bz2} {url}')
        if ret != 0:
            print(f'    ⚠️  Download failed for {name}, skipping.')
            continue
    
    fsize = os.path.getsize(local_bz2)
    if fsize < 1000:
        print(f'    ⚠️  {name} dump too small ({fsize} bytes), skipping.')
        os.remove(local_bz2)
        continue

    print(f'Parsing {name} ({fsize/1024:.1f} KB)...')
    source_docs = []
    try:
        with bz2.open(local_bz2, 'rt', encoding='utf-8') as f:
            context = iter(ET.iterparse(f, events=('start', 'end')))
            event, root = next(context)
            for event, elem in context:
                tag = elem.tag.split('}')[-1] if '}' in elem.tag else elem.tag
                if event != 'end' or tag != 'page': continue
                ns_el = elem.find('{http://www.mediawiki.org/xml/export-0.10/}ns')
                if ns_el is None: ns_el = elem.find('ns')
                if ns_el is not None and ns_el.text and ns_el.text.strip() != '0':
                    elem.clear(); root.clear(); continue
                rev = None
                for tag_name in ['revision', '{http://www.mediawiki.org/xml/export-0.10/}revision']:
                    rev = elem.find(tag_name)
                    if rev is not None: break
                raw = ''
                if rev is not None:
                    for tag_name in ['text', '{http://www.mediawiki.org/xml/export-0.10/}text']:
                        te = rev.find(tag_name)
                        if te is not None and te.text:
                            raw = te.text; break
                elem.clear(); root.clear()
                if not raw.strip(): continue
                cleaned = clean_wiki(raw)
                if len(cleaned) >= MIN_CHARS and devanagari_ratio(cleaned) >= MIN_DEV_RATIO and latin_ratio(cleaned) <= MAX_LATIN_RATIO:
                    hi = text_hash_int(cleaned)
                    if hi in existing_hashes: continue
                    existing_hashes.add(hi)
                    source_docs.append({'doc_id': f'{name}_{hi:x}', 'source': f'Hindi {name}', 'collection_type': 'manual', 'language': 'hindi', 'text': cleaned, 'clean_text_characters': len(cleaned), 'text_sha256': f'{hi:x}'})
                    new_manual_chars += len(cleaned); new_manual_docs += 1
    except Exception as e:
        print(f'    ⚠️  Error parsing {name}: {e}')
    
    # Save shards for this source
    if source_docs:
        for i in range(0, len(source_docs), SHARD_SIZE):
            batch = source_docs[i:i+SHARD_SIZE]
            sf = f'{MANUAL_DIR}/manual_{name}_shard_{shard_idx:04d}.jsonl'
            with open(sf, 'w', encoding='utf-8') as out:
                for d in batch: out.write(json.dumps(d, ensure_ascii=False) + '\n')
            print(f'  ✅ {Path(sf).name}: {len(batch):,} docs')
            shard_idx += 1
    print(f'  {name}: {len(source_docs):,} docs total')
    gc.collect()

print(f'\n📊 Wiki auxiliary: {new_manual_docs:,} new docs, {new_manual_chars:,} chars (~{int(new_manual_chars / CHARS_PER_TOKEN):,} tokens)')

---
## Cell 3: Large-Scale Hindi News Crawl (Sitemap + Deep Archive Pagination)

In [ ]:
import gzip

print('\n🌐 Phase A: Discovering real article URLs...')
remaining_chars = max(chars_needed - new_manual_chars, 30_000_000)
remaining_tokens = int(remaining_chars / CHARS_PER_TOKEN)
print(f'   Still need: ~{remaining_chars:,} chars (~{remaining_tokens:,} tokens)')

headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/125.0.0.0 Safari/537.36'}

# ── Sitemap discovery ──
SITEMAP_SOURCES = [
    'https://www.amarujala.com/sitemap.xml',
    'https://ndtv.in/sitemap.xml',
    'https://www.livehindustan.com/sitemap.xml',
    'https://navbharattimes.indiatimes.com/sitemap.cms',
    'https://www.jagran.com/sitemap.xml',
    'https://www.bhaskar.com/sitemap.xml',
    'https://www.patrika.com/sitemap.xml',
    'https://www.abplive.com/sitemap.xml',
    'https://zeenews.india.com/hindi/sitemap.xml',
    'https://www.tv9hindi.com/sitemap.xml',
]

def fetch_sitemap_urls(sitemap_url, depth=0, max_urls=10000):
    if depth > 2: return []
    urls = []
    try:
        r = requests.get(sitemap_url, headers=headers, timeout=15)
        if r.status_code != 200: return urls
        content = r.content
        if sitemap_url.endswith('.gz'):
            try: content = gzip.decompress(content)
            except: return urls
        soup = BeautifulSoup(content, 'xml')
        # Sitemap index?
        sitemap_tags = soup.find_all('sitemap')
        if sitemap_tags:
            for sm in sitemap_tags[:20]:
                loc = sm.find('loc')
                if loc and loc.text:
                    child = fetch_sitemap_urls(loc.text.strip(), depth+1, max_urls=3000)
                    urls.extend(child)
                    if len(urls) >= max_urls: break
        else:
            for tag in soup.find_all('url'):
                loc = tag.find('loc')
                if loc and loc.text: urls.append(loc.text.strip())
                if len(urls) >= max_urls: break
            if not urls:
                for loc in soup.find_all('loc'):
                    if loc.text and 'http' in loc.text: urls.append(loc.text.strip())
                    if len(urls) >= max_urls: break
    except Exception as e:
        pass  # Silently skip broken sitemaps
    return urls

# RSS feeds
RSS_FEEDS = [
    'https://www.amarujala.com/rss/india-news.xml',
    'https://www.amarujala.com/rss/business.xml',
    'https://www.amarujala.com/rss/world.xml',
    'https://www.amarujala.com/rss/education.xml',
    'https://feeds.feedburner.com/ndtvkhabar',
    'https://www.jagran.com/rss/national-news.xml',
    'https://www.jagran.com/rss/business.xml',
    'https://www.jansatta.com/feed/',
    'https://hindi.oneindia.com/rss/india-news-fb.xml',
    'https://www.abplive.com/feed',
    'https://www.livehindustan.com/rss/national',
    'https://www.livehindustan.com/rss/business',
]

def fetch_rss_urls(rss_url):
    urls = []
    try:
        r = requests.get(rss_url, headers=headers, timeout=10)
        if r.status_code == 200:
            soup = BeautifulSoup(r.content, 'xml')
            for link in soup.find_all('link'):
                if link.text and 'http' in link.text: urls.append(link.text.strip())
            for guid in soup.find_all('guid'):
                if guid.text and 'http' in guid.text: urls.append(guid.text.strip())
    except: pass
    return list(set(urls))

# Deep archive pagination (many more pages)
ARCHIVE_PAGES = []
# Amar Ujala: 6 categories × 100 pages = 600 pages
for cat in ['india-news', 'world', 'education', 'business', 'sports', 'entertainment']:
    ARCHIVE_PAGES.extend([f'https://www.amarujala.com/{cat}?page={i}' for i in range(1, 101)])
# OneIndia Hindi: 3 categories × 100 pages
for cat in ['india', 'international', 'business']:
    ARCHIVE_PAGES.extend([f'https://hindi.oneindia.com/news/{cat}/?page-no={i}' for i in range(1, 101)])
# Jansatta: 4 categories × 80 pages
for cat in ['national', 'rajniti', 'business', 'entertainment']:
    ARCHIVE_PAGES.extend([f'https://www.jansatta.com/{cat}/page/{i}/' for i in range(1, 81)])
# LiveHindustan: 3 categories × 100 pages
for cat in ['national', 'business', 'entertainment']:
    ARCHIVE_PAGES.extend([f'https://www.livehindustan.com/{cat}/page-{i}' for i in range(1, 101)])
# NDTV Hindi
for cat in ['india', 'world', 'business', 'entertainment']:
    ARCHIVE_PAGES.extend([f'https://ndtv.in/{cat}/page-{i}' for i in range(1, 51)])
# ABP Live
for cat in ['news', 'business', 'sports']:
    ARCHIVE_PAGES.extend([f'https://www.abplive.com/{cat}/page/{i}' for i in range(1, 51)])

print(f'  Total archive pages to scrape: {len(ARCHIVE_PAGES)}')

def extract_links_from_page(page_url):
    urls = []
    try:
        domain = urlparse(page_url).netloc
        r = requests.get(page_url, headers=headers, timeout=10)
        if r.status_code == 200:
            soup = BeautifulSoup(r.text, 'html.parser')
            for a in soup.find_all('a', href=True):
                href = a['href']
                if not href.startswith('http'): href = urljoin(page_url, href)
                if domain not in href: continue
                parts = href.rstrip('/').split('/')
                if len(parts) >= 5 and not any(x in href for x in ['?page', 'page-no=', '/page/', '/tag/', '/category/', '/author/', '#', 'javascript:', '/rss/', '/sitemap', '/feed']):
                    urls.append(href)
    except: pass
    return list(set(urls))

# ── Collect URLs ──
all_crawl_urls = set()

# 1. Sitemaps
print('  Fetching sitemaps...')
for surl in SITEMAP_SOURCES:
    discovered = fetch_sitemap_urls(surl)
    if discovered:
        all_crawl_urls.update(discovered)
        print(f'    📍 {urlparse(surl).netloc}: {len(discovered):,} URLs')
print(f'  After sitemaps: {len(all_crawl_urls):,}')

# 2. RSS
print('  Fetching RSS feeds...')
for rss_url in RSS_FEEDS:
    discovered = fetch_rss_urls(rss_url)
    all_crawl_urls.update(discovered)
print(f'  After RSS: {len(all_crawl_urls):,}')

# 3. Archive pages (parallel)
print(f'  Scraping {len(ARCHIVE_PAGES)} archive pages (15 workers)...')
with ThreadPoolExecutor(max_workers=15) as executor:
    futures = {executor.submit(extract_links_from_page, p): p for p in ARCHIVE_PAGES}
    done = 0
    for future in as_completed(futures):
        all_crawl_urls.update(future.result())
        done += 1
        if done % 200 == 0:
            print(f'    ... {done}/{len(ARCHIVE_PAGES)} pages scraped, {len(all_crawl_urls):,} URLs so far')
print(f'  After archives: {len(all_crawl_urls):,}')

crawl_urls = list(all_crawl_urls)
random.shuffle(crawl_urls)
print(f'\n✅ Total unique URLs: {len(crawl_urls):,}')

---
## Cell 3b: Execute Parallel Crawl

In [ ]:
print(f'🌐 Crawling {len(crawl_urls):,} URLs with 30 workers...')
crawl_docs = []
crawl_chars_total = 0

def fetch_hindi_url(url):
    try:
        r = requests.get(url, headers=headers, timeout=8)
        if r.status_code == 200:
            text = trafilatura.extract(r.text, include_tables=False)
            if text:
                cp = clean_web_text(text)
                if len(cp) >= MIN_CHARS and devanagari_ratio(cp) >= MIN_DEV_RATIO and latin_ratio(cp) <= MAX_LATIN_RATIO:
                    hi = text_hash_int(cp)
                    if hi not in existing_hashes:
                        return {'doc_id': f'{hi:x}', 'url': url, 'source': 'Hindi News Crawl', 'source_type': 'news_web', 'collection_type': 'manual', 'language': 'hindi', 'text': cp, 'clean_text_characters': len(cp), 'text_sha256': f'{hi:x}', 'hash_int': hi}
    except: pass
    return None

CRAWL_CHAR_TARGET = max(remaining_chars, 30_000_000)

with ThreadPoolExecutor(max_workers=30) as executor:
    futures = {executor.submit(fetch_hindi_url, u): u for u in crawl_urls}
    done_count = 0
    for future in as_completed(futures):
        doc = future.result()
        done_count += 1
        if doc:
            existing_hashes.add(doc.pop('hash_int'))
            crawl_docs.append(doc)
            crawl_chars_total += doc['clean_text_characters']
            if len(crawl_docs) >= SHARD_SIZE:
                sf = f'{MANUAL_DIR}/manual_news_shard_{shard_idx:04d}.jsonl'
                with open(sf, 'w', encoding='utf-8') as out:
                    for d in crawl_docs: out.write(json.dumps(d, ensure_ascii=False) + '\n')
                print(f'  ✅ {Path(sf).name}: {len(crawl_docs):,} docs | cumulative: {crawl_chars_total:,} chars (~{int(crawl_chars_total/CHARS_PER_TOKEN):,} tokens)')
                crawl_docs = []; shard_idx += 1; gc.collect()
        if done_count % 5000 == 0:
            print(f'  ... {done_count:,}/{len(crawl_urls):,} URLs processed | {crawl_chars_total:,} chars')
        if crawl_chars_total >= CRAWL_CHAR_TARGET:
            print(f'\n🎯 Target reached ({crawl_chars_total:,} chars)!')
            executor.shutdown(wait=False, cancel_futures=True)
            break

if crawl_docs:
    sf = f'{MANUAL_DIR}/manual_news_shard_{shard_idx:04d}.jsonl'
    with open(sf, 'w', encoding='utf-8') as out:
        for d in crawl_docs: out.write(json.dumps(d, ensure_ascii=False) + '\n')
    print(f'  ✅ Final shard {Path(sf).name}: {len(crawl_docs):,} docs')

total_new_chars = new_manual_chars + crawl_chars_total
grand_manual_chars = existing_manual_chars + total_new_chars

print('\n' + '=' * 70)
print('MANUAL COLLECTION SUMMARY')
print('=' * 70)
print(f'  Previously:    {existing_manual_chars:,} chars (~{int(existing_manual_chars/CHARS_PER_TOKEN):,} tokens)')
print(f'  Wiki dumps:    +{new_manual_chars:,} chars (~{int(new_manual_chars/CHARS_PER_TOKEN):,} tokens)')
print(f'  News crawl:    +{crawl_chars_total:,} chars (~{int(crawl_chars_total/CHARS_PER_TOKEN):,} tokens)')
print(f'  GRAND TOTAL:   {grand_manual_chars:,} chars (~{int(grand_manual_chars/CHARS_PER_TOKEN):,} tokens)')
print('=' * 70)

---
## Cell 4: Remount Drive & Re-Create Splits

In [ ]:
import random, shutil

# ── Remount Drive (fixes 'Transport endpoint not connected') ──
print('🔄 Remounting Google Drive to ensure stable connection...')
try:
    drive.flush_and_unmount()
    time.sleep(2)
except: pass
drive.mount('/content/drive', force_remount=True)
time.sleep(3)
print('✅ Drive remounted.')

# ── Verify Drive is accessible ──
test_shards = list(Path(NONWEB_DIR).glob('*.jsonl'))
test_manual = list(Path(MANUAL_DIR).glob('*.jsonl'))
print(f'Drive check: {len(test_manual)} manual shards, {len(test_shards)} automated shards visible.')

LOCAL_MANUAL = '/content/local_manual'
LOCAL_AUTO = '/content/local_auto'
LOCAL_SPLITS = '/content/splits'
for d in [LOCAL_MANUAL, LOCAL_AUTO, LOCAL_SPLITS]:
    if os.path.exists(d): shutil.rmtree(d)
    os.makedirs(d)

# ── Copy shards to local SSD with retry logic ──
def copy_with_retry(src_dir, dst_dir, pattern='*.jsonl', max_retries=3):
    files = sorted(Path(src_dir).glob(pattern))
    copied, failed = 0, 0
    for f in files:
        for attempt in range(max_retries):
            try:
                shutil.copy2(str(f), dst_dir)
                copied += 1
                break
            except Exception as e:
                if attempt == max_retries - 1:
                    print(f'    ⚠️  Failed to copy {f.name}: {e}')
                    failed += 1
                else:
                    time.sleep(1)
    return copied, failed

print('Copying manual shards...')
mc, mf = copy_with_retry(MANUAL_DIR, LOCAL_MANUAL)
print(f'  Copied {mc} manual shards ({mf} failed)')

print('Copying automated shards...')
ac, af = copy_with_retry(NONWEB_DIR, LOCAL_AUTO)
print(f'  Copied {ac} automated shards ({af} failed)')

if ac == 0:
    print('\n⚠️  ZERO automated shards copied! Drive may be disconnected.')
    print('    Run this in a new cell: drive.mount("/content/drive", force_remount=True)')
    print('    Then re-run this cell.')
    raise RuntimeError('No automated shards available')

# ── Create splits ──
random.seed(42)
out = {s: open(f'{LOCAL_SPLITS}/{s}.txt', 'w', encoding='utf-8') for s in ['train', 'val', 'test']}
jout = {s: open(f'{LOCAL_SPLITS}/{s}.jsonl', 'w', encoding='utf-8') for s in ['train', 'val', 'test']}
counts = {'train': 0, 'val': 0, 'test': 0}
manual_char_total, auto_char_total = 0, 0
skipped_lines = 0

print('\nIngesting manual shards...')
for sf in sorted(Path(LOCAL_MANUAL).glob('*.jsonl')):
    with open(sf, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip(): continue
            try:
                doc = json.loads(line)
            except (json.JSONDecodeError, UnicodeDecodeError):
                skipped_lines += 1; continue
            t = doc.get('text', '')
            if not t.strip(): continue
            r = random.random()
            s = 'train' if r < 0.90 else ('val' if r < 0.95 else 'test')
            jout[s].write(line); out[s].write(t.strip() + '\n\n')
            counts[s] += 1; manual_char_total += len(t)

MAX_AUTO_CHARS = int(400_000_000 * CHARS_PER_TOKEN)
print(f'Ingesting automated shards (up to {MAX_AUTO_CHARS:,} chars)...')
for sf in sorted(Path(LOCAL_AUTO).glob('*.jsonl')):
    if auto_char_total >= MAX_AUTO_CHARS: break
    with open(sf, 'r', encoding='utf-8') as f:
        for line in f:
            if auto_char_total >= MAX_AUTO_CHARS: break
            if not line.strip(): continue
            try:
                doc = json.loads(line)
            except (json.JSONDecodeError, UnicodeDecodeError):
                skipped_lines += 1; continue
            t = doc.get('text', '')
            if not t.strip(): continue
            r = random.random()
            s = 'train' if r < 0.90 else ('val' if r < 0.95 else 'test')
            jout[s].write(line); out[s].write(t.strip() + '\n\n')
            counts[s] += 1; auto_char_total += len(t)

for f in out.values(): f.close()
for f in jout.values(): f.close()

tot = manual_char_total + auto_char_total
print('\n' + '=' * 70)
print('SPLIT COMPLETE')
print(f'  Manual:    ~{int(manual_char_total / CHARS_PER_TOKEN):,} tokens ({manual_char_total / max(1, tot) * 100:.1f}%)')
print(f'  Automated: ~{int(auto_char_total / CHARS_PER_TOKEN):,} tokens ({auto_char_total / max(1, tot) * 100:.1f}%)')
print(f'  Total:     ~{int(tot / CHARS_PER_TOKEN):,} tokens')
for s in ['train', 'val', 'test']: print(f'  {s}: {counts[s]:,} docs')
if skipped_lines: print(f'  ⚠️  Skipped {skipped_lines} corrupt lines')
print('=' * 70)

print('\nSyncing splits to Drive...')
!cp -r {LOCAL_SPLITS}/* {SPLITS_DIR}/
print('✅ Splits synced.')

---
## Cell 5: Re-Train SentencePiece BPE Tokenizer (8k)

In [ ]:
MPREFIX = f'{TOKENIZER_DIR}/hindi_bpe_8k'
TRAIN_TXT = f'{LOCAL_SPLITS}/train.txt'
mb = os.path.getsize(TRAIN_TXT) / (1024 * 1024)
print(f'Training BPE 8k on {mb:.1f} MB...')

spm.SentencePieceTrainer.train(
    input=TRAIN_TXT, model_prefix=MPREFIX, vocab_size=8000,
    model_type='bpe', character_coverage=0.9995, num_threads=4,
    max_sentence_length=8192, input_sentence_size=500_000,
    shuffle_input_sentence=True,
    pad_id=0, unk_id=1, bos_id=2, eos_id=3,
    pad_piece='<pad>', unk_piece='<unk>', bos_piece='<bos>', eos_piece='<eos>',
    split_by_unicode_script=True, split_by_whitespace=True,
    split_digits=True, byte_fallback=True,
)
sp = spm.SentencePieceProcessor()
sp.load(f'{MPREFIX}.model')
print(f'\n✅ Vocab: {sp.get_piece_size():,}')
print(sp.encode('भारत एक प्राचीन, सांस्कृतिक और संप्रभु लोकतांत्रिक गणराज्य है।', out_type=str))

---
## Cell 6: Final Intrinsic Benchmark & Corpus Summary

In [ ]:
VAL_TXT = f'{LOCAL_SPLITS}/val.txt'
COMBINING_MARKS = set(chr(c) for c in range(0x0901, 0x0904)) | set(chr(c) for c in range(0x093E, 0x094E)) | {chr(0x0962), chr(0x0963)}

def count_broken_graphemes(tokens):
    return sum(1 for tok in tokens if tok.lstrip(' ') and tok.lstrip(' ')[0] in COMBINING_MARKS)

val_sentences = []
with open(VAL_TXT, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line and len(line) > 20: val_sentences.append(line)
        if len(val_sentences) >= 5000: break

sp = spm.SentencePieceProcessor()
sp.load(f'{MPREFIX}.model')

total_words, total_tokens, total_chars, total_broken = 0, 0, 0, 0
token_freqs = Counter()
for text in val_sentences:
    words = text.split(); tokens = sp.encode(text, out_type=str); ids = sp.encode(text, out_type=int)
    total_words += len(words); total_tokens += len(tokens); total_chars += len(text)
    total_broken += count_broken_graphemes(tokens); token_freqs.update(ids)

fertility = total_tokens / max(1, total_words)
chars_per_tok = total_chars / max(1, total_tokens)
broken_rate = (total_broken / max(1, total_tokens)) * 100
probs = [c / total_tokens for c in token_freqs.values()]
shannon = -sum(p * math.log2(p) for p in probs if p > 0)
renyi = -math.log2(sum(p**2 for p in probs))

print('\n' + '=' * 75)
print('INTRINSIC TOKENIZER METRICS (Hindi BPE 8k)')
print('=' * 75)
print(f'  Fertility:          {fertility:.2f} tokens/word')
print(f'  Characters/Token:   {chars_per_tok:.2f}')
print(f'  Broken Matra Rate:  {broken_rate:.3f}%')
print(f'  Shannon Entropy:    {shannon:.2f} bits')
print(f'  Rényi Entropy:      {renyi:.2f} bits')
print('=' * 75)

token_counts = {}
for sn in ['train', 'val', 'test']:
    tt, td = 0, 0
    with open(f'{LOCAL_SPLITS}/{sn}.txt', 'r', encoding='utf-8') as f:
        buf = []
        for line in f:
            if line.strip(): buf.append(line.strip())
            elif buf: tt += len(sp.encode(' '.join(buf))); td += 1; buf = []
        if buf: tt += len(sp.encode(' '.join(buf))); td += 1
    token_counts[sn] = {'docs': td, 'tokens': tt}
    print(f'  {sn.upper()}: {td:,} docs → {tt:,} tokens')

TOTAL_TOKENS = sum(v['tokens'] for v in token_counts.values())
manual_share = round((manual_char_total / max(1, tot)) * 100, 2)

summary = {'language': LANGUAGE, 'manual_chars': manual_char_total, 'automated_chars': auto_char_total, 'total_chars': tot, 'manual_percentage': manual_share, 'token_counts': token_counts, 'total_tokens': TOTAL_TOKENS, 'vocab_size': sp.get_piece_size()}
with open(f'{DRIVE_BASE}/corpus_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(f'\n🎉 FINAL AUDIT:')
print(f'  Total Tokens:      {TOTAL_TOKENS:,} ({"✅ PASSED" if TOTAL_TOKENS >= 500_000_000 else "⚠️ SHORT of 500M"})')
print(f'  Manual Share:      {manual_share:.2f}% ({"✅ PASSED" if manual_share >= 20.0 else "⚠️ SHORT of 20%"})')